# 02 · Predictive Model — Random Forest Classifier

## Modelling Strategy

We train a **Random Forest** classifier to predict 90-day attrition risk.  
Choice rationale:
- Handles mixed feature types without scaling
- Robust to multicollinearity (engagement ↔ satisfaction)
- Feature importances give explainable output for HR audiences
- Comparable AUC to gradient boosting on tabular HR data at this sample size

**Target variable:** `attrited` (1 = left within the period, 0 = retained)  
**Threshold:** 0.40 (lower than 0.50 to favour recall — catching a flight-risk employee is worth more than a false positive in most HR use cases)


In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, roc_curve)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/employee_data.csv')

In [ ]:
# --- ENCODE CATEGORICALS ---
cat_cols = ['gender','department']
le = {}
df_enc = df.copy()
for c in cat_cols:
    le[c] = LabelEncoder()
    df_enc[c] = le[c].fit_transform(df[c])

FEATURES = [
    'age','job_level','tenure_years','compa_ratio','performance_rating',
    'engagement_score','satisfaction_score','overtime_hours_per_week',
    'years_since_last_promotion','work_life_balance','training_hours_last_year',
    'manager_tenure_years','num_prior_companies','commute_miles',
    'department','gender'
]
TARGET = 'attrited'

X = df_enc[FEATURES]
y = df_enc[TARGET]
print(f"Features: {len(FEATURES)} | Target class balance: {y.value_counts().to_dict()}")

In [ ]:
# --- TRAIN / TEST SPLIT (stratified) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# --- MODEL ---
model = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    class_weight='balanced',   # handles class imbalance
    random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)
print("Model trained.")

In [ ]:
# --- EVALUATION ---
y_prob = model.predict_proba(X_test)[:,1]
y_pred = (y_prob >= 0.40).astype(int)

auc = roc_auc_score(y_test, y_prob)
cv  = cross_val_score(model, X, y, cv=StratifiedKFold(5), scoring='roc_auc')

print(f"ROC-AUC  (test):          {auc:.3f}")
print(f"ROC-AUC  (5-fold CV):     {cv.mean():.3f} ± {cv.std():.3f}")
print()
print(classification_report(y_test, y_pred, target_names=['Retained','Attrited']))

In [ ]:
# --- ROC CURVE ---
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(fpr, tpr, color='#2F81F7', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Attrition Predictor')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../outputs/figures/fig4_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- FEATURE IMPORTANCE ---
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Top 10 predictors of attrition:")
print(importances.head(10).map('{:.4f}'.format))

In [ ]:
# Save model
import pickle
with open('../models/rf_attrition_model.pkl','wb') as f:
    pickle.dump({'model': model, 'encoders': le, 'features': FEATURES}, f)
print("Model saved to models/rf_attrition_model.pkl")